# Introduction à PySpark – Traitement CSV

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanouar/1to1code/blob/main/pySpark/notebooks/02_traitement_csv.ipynb)

Notebook conçu pour **Google Colab**.  
On explore la lecture de CSV, les transformations et les agrégations avec PySpark.

> **SparkUI** : accessible après la cellule de setup via le lien affiché.


In [2]:
# Installation + setup (Colab)
!pip install -q pyspark findspark
!git clone --filter=blob:none --sparse https://github.com/tanouar/1to1code.git -q \
  && cd 1to1code && git sparse-checkout set pySpark -q

import os, sys
PYSPARK_TOOLS = os.path.join('/content', '1to1code', 'pySpark')
if PYSPARK_TOOLS not in sys.path:
    sys.path.insert(0, PYSPARK_TOOLS)

from sparktools import setup_colab, download_iris

spark, monitor, helper = setup_colab("PySpark - Traitement CSV")


## 1. Créer un DataFrame

On crée un DataFrame manuellement avec `createDataFrame`.  
→ SparkUI : pas de job déclenché (pas d'action ici).


In [3]:
data = [("Alice", 25), ("Bob", 30), ("Charlie", 35)]
df_test = spark.createDataFrame(data, ["nom", "age"])
df_test.show()


## 2. Lire un CSV

`download_iris()` télécharge le dataset Iris depuis UCI.  
→ SparkUI : le `read.csv` déclenche un job de scan du fichier.


In [4]:
csv_path = download_iris()   # télécharge dans /content/iris.csv

df = spark.read.csv(csv_path, header=False, inferSchema=True)
noms_colonnes = ["sepal_length", "sepal_width", "petal_length", "petal_width", "class"]
df = df.toDF(*noms_colonnes)
df.show(5)


In [5]:
df.printSchema()
df.show(5)


## 3. Explorer les données

`describe()` déclenche un job de calcul sur toutes les colonnes numériques.  
→ SparkUI : observez les **stages** de ce job.


In [6]:
monitor.execute_and_monitor(
    lambda: df.describe().show(),
    "Statistiques descriptives"
)


## 4. Agrégations

Regroupement par espèce avec `groupBy` + `agg`.  
→ SparkUI : observez le plan d'exécution dans l'onglet **SQL / DataFrame**.


In [7]:
from pyspark.sql.functions import avg, max, min, count

monitor.execute_and_monitor(
    lambda: df.groupBy("class")
        .agg(
            count("*").alias("nb"),
            avg("petal_length").alias("petal_length_avg"),
            max("petal_width").alias("petal_width_max")
        )
        .orderBy("petal_length_avg", ascending=False)
        .show(),
    "Agrégation par espèce"
)


## 5. Gros volume (100 millions de lignes)

On génère 100 millions de lignes pour mesurer la puissance de Spark.  
→ SparkUI : observez le nombre de **Tasks** exécutées en parallèle.


In [8]:
from pyspark.sql.functions import col, rand, when

df_big = spark.range(0, 100_000_000).toDF("id") \
    .withColumn("valeur", (rand() * 1000).cast("int")) \
    .withColumn("categorie", when(col("valeur") < 300, "Faible")
                              .when(col("valeur") < 700, "Moyen")
                              .otherwise("Élevé"))

monitor.execute_and_monitor(lambda: df_big.count(), "Count 100M")
monitor.execute_and_monitor(
    lambda: df_big.groupBy("categorie").count().show(),
    "Agrégation 100M"
)
monitor.show_history()
